# Análisis de Precipitación CHIRPS para El Salvador

Este cuaderno recupera datos diarios de precipitación desde ERDDAP (CHIRPS v2.0), calcula acumulados mensuales, una climatología mensual y las anomalías correspondientes para El Salvador.

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

## Descarga de datos diarios

Ajusta `start_year` y `end_year` si deseas analizar un periodo diferente.

In [ ]:
start_year = 2000
end_year = 2023

lat_min, lat_max = 13.0, 14.5
lon_min, lon_max = -90.0, -87.0

start_date = f"{start_year}-01-01"
end_date = f"{end_year}-12-31"

base_url = (
    "https://coastwatch.pfeg.noaa.gov/erddap/griddap/chirps20GlobalDaily.nc"
    "?precip[({start}):({end})][({lat_min}):({lat_max})][({lon_min}):({lon_max})]"
)

url = base_url.format(
    start=start_date,
    end=end_date,
    lat_min=lat_min,
    lat_max=lat_max,
    lon_min=lon_min,
    lon_max=lon_max,
)

print(url)

ds = xr.open_dataset(url)
precip_daily = ds['precip']
precip_daily

## Promedio espacial y extracción de subregiones

El siguiente bloque calcula el promedio espacial sobre El Salvador. Puedes modificar `precip_agricola` para analizar subregiones agrícolas específicas utilizando `.sel` con rangos de latitud/longitud más estrechos.

In [ ]:
precip_es = precip_daily.mean(dim=['latitude', 'longitude'])

# Ejemplo de selección de una subregión agrícola (ajusta los valores según sea necesario)
subregion = precip_daily.sel(latitude=slice(14.2, 13.5), longitude=slice(-89.5, -88.5))
precip_agricola = subregion.mean(dim=['latitude', 'longitude'])

precip_es.to_pandas().head()

## Agregación mensual, climatología y anomalías

1. Se convierten los totales diarios a acumulados mensuales.
2. Se calcula la climatología media mensual del periodo completo.
3. Se obtienen las anomalías mensuales restando la climatología al acumulado mensual correspondiente.

In [ ]:
precip_mensual = precip_es.resample(time='1M').sum()
climatologia_mensual = precip_mensual.groupby('time.month').mean('time')

anomalias = precip_mensual.groupby('time.month') - climatologia_mensual

precip_mensual_df = precip_mensual.to_pandas()
climatologia_df = climatologia_mensual.to_pandas()
anomalias_df = anomalias.to_pandas()

precip_mensual_df.head()

## Visualización de resultados

Las siguientes figuras muestran:

- Serie temporal mensual acumulada.
- Climatología mensual.
- Anomalías mensuales respecto a la climatología.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 14), sharex=False)

precip_mensual_df.plot(ax=ax[0], color='tab:blue')
ax[0].set_title('Precipitación mensual acumulada (El Salvador)')
ax[0].set_ylabel('mm')
ax[0].grid(True, linestyle='--', alpha=0.4)

climatologia_df.plot(kind='bar', ax=ax[1], color='tab:green')
ax[1].set_title('Climatología mensual de precipitación')
ax[1].set_ylabel('mm')
ax[1].grid(True, axis='y', linestyle='--', alpha=0.4)

anomalias_df.plot(ax=ax[2], color='tab:red')
ax[2].axhline(0, color='black', linewidth=1)
ax[2].set_title('Anomalías mensuales de precipitación')
ax[2].set_ylabel('mm respecto a la climatología')
ax[2].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## Observaciones preliminares

- Los máximos de precipitación se concentran típicamente en los meses de mayo a octubre, coherentes con la estación lluviosa centroamericana.
- Los mínimos se observan entre diciembre y marzo, reflejando la estación seca.
- Identifica años con anomalías positivas destacadas (p. ej., posibles eventos de tormentas tropicales) y negativos (sequías) comparando las barras de la climatología con la serie mensual.
- Se recomienda complementar este análisis con información agrícola local para interpretar impactos sobre cultivos clave.